### Import Movie Lens Data for Recommendation System

##### This combines movielens, letterboxd, and tmdb data

In [2]:
import pandas as pd

In [2]:
ml_ratings = pd.read_csv("../data/movielens/ratings.csv")
ml_movies = pd.read_csv("../data/movielens/movies.csv")
ml_links = pd.read_csv("../data/movielens/links.csv")


In [3]:
ml32M_with_ratings = pd.read_parquet("../data/movielens_ratings_w_tmdbId.parquet")

In [ ]:
movies_with_features = pd.read_parquet("../data/movie_with_features.parquet")

In [ ]:
movies_w_features_and_ratings = pd.merge(
    ml32M_with_ratings,

In [4]:
ml32M_with_ratings.head()

,userId,tmdbId,rating
0,1,4584.0,4.0
1,1,451.0,1.0
2,1,902.0,2.0
3,1,37557.0,5.0
4,1,63.0,5.0


##### Popularity Based Recommendation System

In [33]:
# Get movies with highest average ratings
combined_ratings_w_name = combined_ratings.merge(ml_movies[["movieId", "title"]], on="tmdbId")

KeyError: 'tmdbId'

In [31]:
# Get movies with more than 10 ratings
movie_thresh = combined_ratings_w_name.groupby("movieId")["userId"].count().sort_values(ascending=False)
movie_thresh = movie_thresh[movie_thresh >= 10]

In [32]:
movie_thresh.shape

(2272,)

In [28]:
movie_thresh.head()

movieId
356     329
318     318
296     308
593     280
2571    278
Name: userId, dtype: int64

In [34]:
combined_ratings_final = combined_ratings_w_name[combined_ratings_w_name["movieId"].isin(movie_thresh.index)]
combined_ratings_final.head()

,userId,movieId,rating,title
0,1,1,4.0,Toy Story (1995)
1,1,3,4.0,Grumpier Old Men (1995)
2,1,6,4.0,Heat (1995)
3,1,47,5.0,Seven (a.k.a. Se7en) (1995)
4,1,50,5.0,"Usual Suspects, The (1995)"


In [36]:
# Get movies with most number of watchers
movie_freq = combined_ratings_final.groupby("title")["userId"].count().sort_values(ascending=False)

In [37]:
# Get highest rated movies with at least 10 ratings
highest_rated_movies = combined_ratings_final.groupby("title").mean().sort_values(("rating"), ascending=False)

In [ ]:
popular_movies = pd.DataFrame({"avg_rating": highest_rated_movies["rating"], "num_ratings": movie_freq})

,avg_rating,num_ratings
title,,
"'burbs, The (1989)",3.176471,17
(500) Days of Summer (2009),3.666667,42
10 Cloverfield Lane (2016),3.678571,14
10 Things I Hate About You (1999),3.527778,54
"10,000 BC (2008)",2.705882,17
101 Dalmatians (1996),3.074468,47
101 Dalmatians (One Hundred and One Dalmatians) (1961),3.431818,44
12 Angry Men (1957),4.155172,58
12 Years a Slave (2013),3.625000,16


In [39]:
popular_movies.sort_values("avg_rating", ascending=False).head(10)

,avg_rating,num_ratings
title,,
Secrets & Lies (1996),4.590909,11
Guess Who's Coming to Dinner (1967),4.545455,11
Paths of Glory (1957),4.541667,12
"Streetcar Named Desire, A (1951)",4.475000,20
"Celebration, The (Festen) (1998)",4.458333,12
Ran (1985),4.433333,15
"Shawshank Redemption, The (1994)",4.429245,318
His Girl Friday (1940),4.392857,14
All Quiet on the Western Front (1930),4.350000,10


In [40]:
# Create a function to get recommendations based on rating with a minimum number of ratings
def top_movies(min_ratings=10, top_n=10):
    popular_movies_filtered = popular_movies[popular_movies["num_ratings"] >= min_ratings]
    return popular_movies_filtered.sort_values("avg_rating", ascending=False).head(top_n)

In [ ]:
# Get movie recommendations with at least 100 ratings and return the top 10
top_movies(100, 10)

,avg_rating,num_ratings
title,,
"Shawshank Redemption, The (1994)",4.429245,318
"Godfather, The (1972)",4.289062,192
Fight Club (1999),4.271689,219
"Godfather: Part II, The (1974)",4.259690,129
Goodfellas (1990),4.251969,127
"Departed, The (2006)",4.250000,108
"Usual Suspects, The (1995)",4.241463,205
Casablanca (1942),4.240000,100
"Dark Knight, The (2008)",4.236667,150
